# ScamTrace: explainable scam-signal demo

This notebook demonstrates ScamTrace's Python heuristic engine using **synthetic examples only**. It shows how a URL or message is screened, how signals contribute to a score, and how common contact details are redacted before a report is stored.

> **Important:** This is an advisory screen, not a definitive verdict or live threat-intelligence service. A low score does not prove that a URL or message is safe. The examples below are fictional. ScamTrace does not fetch, resolve, or open submitted URLs.

## Run it
Open this notebook from the ScamTrace project, or keep the notebook next to the `src/` folder. The code imports the same `src/analysis.py` module used by the project. It needs Python 3.10 or later and uses only the Python standard library.

In [ ]:
from pathlib import Path
import sys

# Find the project source whether this notebook is in the repo root or outputs/.
candidates = [Path.cwd() / 'src', Path.cwd().parent / 'src']
src_dir = next((path for path in candidates if (path / 'analysis.py').is_file()), None)
if src_dir is None:
    raise FileNotFoundError(
        'Could not find src/analysis.py. Open this notebook from the ScamTrace project root.'
    )
sys.path.insert(0, str(src_dir))

from analysis import analyze, redact_personal_data

print(f'Loaded ScamTrace analysis engine from: {src_dir / "analysis.py"}')

## 1. Screen a suspicious URL

This fictional link uses an unrelated domain with security-themed wording. The analyzer inspects the supplied text only; it does not contact the domain.

In [ ]:
sample_url = 'https://account-verify.example.click/login'  # synthetic example
url_result = analyze(sample_url, kind='url')

print('Input:', sample_url)
print(f'Risk: {url_result.risk} | Score: {url_result.score}/100')
print('Signals:')
for signal in url_result.signals:
    print(f'  • {signal}')
print('Advice:', url_result.advice)
print('Disclaimer:', url_result.disclaimer)

## 2. Screen a message

The example imitates a delivery-fee lure. Do not use real messages containing passwords, account details, or other sensitive information in a demo notebook.

In [ ]:
sample_message = (
    'Your parcel is on hold. Pay a small redelivery fee within 24 hours ' 
    'using this link: https://parcel-check.example.click/claim'
)  # synthetic example
message_result = analyze(sample_message, kind='message')

print(f'Risk: {message_result.risk} | Score: {message_result.score}/100')
print('Signals:')
for signal in message_result.signals:
    print(f'  • {signal}')
print('Advice:', message_result.advice)

## 3. Compare several synthetic cases

Scores are heuristic weights capped at 100. Risk labels use the project's configured thresholds. They are not probabilities of fraud.

In [ ]:
examples = [
    ('Shortened URL', 'https://bit.ly/demo-example', 'url'),
    ('Urgent bank message', 'Your account is locked. Verify now with your password.', 'message'),
    ('Plain informational text', 'The community meeting begins at 6 PM on Friday.', 'message'),
]

for label, text, kind in examples:
    result = analyze(text, kind=kind)
    print(f'{label}: {result.risk} ({result.score}/100)')
    print('  ' + '; '.join(result.signals))

## 4. Redact contact details from a report

The backend redacts common email addresses and phone-number-like strings before report storage. This is a basic safeguard, not a guarantee that every personal detail will be detected. Avoid entering real personal data.

In [ ]:
synthetic_report = 'A caller at demo.person@example.invalid asked me to call +1 202 555 0148.'
redacted_report = redact_personal_data(synthetic_report)
print('Before:', synthetic_report)
print('After: ', redacted_report)

## What this demo does not do

- It does not open, fetch, resolve, or reputation-check URLs.
- It does not use machine learning or claim to identify every scam.
- It does not connect to the deployed AWS API or read real user reports.
- The sample text and domains are synthetic; results are for demonstration.

For live use, verify unexpected requests using the organization's official website or a phone number you find independently. Do not rely on a low-risk score as proof of safety.